# CREM: Edad Media de la Población
**Propósito**: Extrae la información sobre la evolución de la edad media de la población según municipios, distritos y secciones censales a partir de la página oficial del CREM (Centro Regional de Estadística de Murcia), procesa y limpia los datos directamente en memoria y los almacena en una tabla Delta Lake en Spark.

In [1]:
import sys
import re
import requests
from bs4 import BeautifulSoup
import pandas as pd
from pyspark.sql import functions as F

sys.path.insert(0, '/tfm/python_notebooks')
from tfm_lib import utils as tfm_utils
from tfm_lib import crem_scrap as crem

In [2]:
# Definir la url de la web
url_base = "https://econet.carm.es/web/crem/inicio/-/crem/sicrem/PM2089//sec27.html"
table_name = "raw.crem.edad_media_poblacion"
column_suffix = "Edad_media_poblacion"

## Descarga y Parseo del Contenido HTML directamente en Memoria
Se realiza una petición HTTP para descargar el contenido HTML de la página oficial del CREM en memoria y procesarlo dinámicamente sin persistir archivos intermedios en el disco local.

In [3]:

# Hacer la solicitud
html = crem.get_html(url_base)

# Convertir HTML a BeautifulSoup object.
soup = BeautifulSoup(html, "html.parser")

table = soup.find("table", id="interiorTablaDatos")

if not table:
    raise Exception("No se encontró la tabla con id 'interiorTablaDatos' en el HTML")

# Obtener cabeceras (años)
thead = table.find("thead", id="cabeceraTablaDatos")
anos_columnas = []
for th in thead.find_all("th"):
    val = th.get_text(strip=True)
    if val:
        anos_columnas.append(val)

print(f"Años detectados: {anos_columnas}")

Años detectados: ['2023', '2022', '2021', '2020', '2019', '2018', '2017', '2016', '2015']


In [4]:
# Parsear filas de datos jerárquicos
tbody = table.find("tbody")
datos_filas = []

mapping_niveles = {
    1: "Región",
    2: "Municipio",
    3: "Distrito",
    4: "Sección"
}

for tr in tbody.find_all("tr"):
    th = tr.find("th", id="r")
    if not th:
        continue
    
    ul = th.find("ul")
    level_num = None
    if ul and ul.has_attr("class"):
        for c in ul["class"]:
            if c.startswith("level"):
                try:
                    level_num = int(c.replace("level", ""))
                except ValueError:
                    pass
                break
    
    label_raw = th.get_text(strip=True)
    label_clean = crem.clean_and_decode(label_raw)
    
    # Extraer código y nombre (ej. 'Abanilla - 30001' o 'Abanilla distrito 01 - 3000101')
    match = re.match(r"^(.*?)\s*-\s*(\d+)$", label_clean)
    if match:
        nombre = match.group(1).strip()
    else:
        nombre = label_clean

    if ',' in nombre: nombre=nombre.split(',')[1]+' '+nombre.split(',')[0]
    
    nivel = mapping_niveles.get(level_num, "Desconocido")
    
    tds = tr.find_all("td", id="d", class_="tbdato")
    values = []
    for td in tds:
        val_str = td.get_text(strip=True).replace(",", ".")
        try:
            val_float = float(val_str) if val_str not in ("", "-", "..") else None
        except ValueError:
            val_float = None
        values.append(val_float)
    
    if len(values) == len(anos_columnas):
        registro = {
            "nombre": crem.normalize_name(nombre),
            "nivel": nivel
        }
        for a, val in zip(anos_columnas, values):
            registro[a] = val
        datos_filas.append(registro)

print(f"Filas parseadas con éxito: {len(datos_filas)}")

Filas parseadas con éxito: 1350


## Integración con PySpark y Delta Lake
Se crea un Spark Session, se inicializa el Spark DataFrame a partir del listado en memoria, se normaliza su esquema con las utilidades de TFM y se guarda como Delta Table en la ruta final.

In [5]:
# Inicializar sesión de Spark usando las utilidades de TFM
spark = tfm_utils.get_spark_session(app_name="CREM_Edad_Media_Poblacion")

# Crear el Spark DataFrame a partir de la lista de registros estructurados
df = (spark.createDataFrame(datos_filas)
          .filter(F.col("nivel").like("Municipio"))
          .drop("nivel")
     )

for anio in anos_columnas:
    df = df.withColumnRenamed(f"{anio}", f"{anio}_{column_suffix}")

# Normalizar las columnas del DataFrame de acuerdo al estilo del TFM (minúsculas, sin acentos, ordenadas)
df_normalized = tfm_utils.normalize_df(df)

# Mostrar los primeros registros de forma estructurada
tfm_utils.display(df_normalized)

,2015_edad_media_poblacion,2016_edad_media_poblacion,2017_edad_media_poblacion,2018_edad_media_poblacion,2019_edad_media_poblacion,2020_edad_media_poblacion,2021_edad_media_poblacion,2022_edad_media_poblacion,2023_edad_media_poblacion,nombre
0,45.6,45.5,45.8,46.3,46.6,46.9,47.1,47.2,47.4,abanilla
1,40.7,40.9,41.2,41.4,41.6,41.8,42.0,42.0,42.2,abaran
2,39.6,39.8,40.1,40.5,40.7,41.1,41.4,41.6,41.8,aguilas
3,41.9,42.7,42.8,42.6,42.9,43.1,43.3,42.9,43.4,albudeite
4,38.2,38.5,38.8,39.2,39.6,39.8,40.1,40.3,40.5,alcantarilla
5,45.8,46.2,46.1,46.6,47.1,47.4,47.3,47.4,47.8,aledo
6,37.5,38.0,38.2,38.5,38.8,38.8,39.0,39.1,39.4,alguazas
7,39.3,39.6,39.7,40.0,40.2,40.6,40.7,40.7,40.9,alhama_de_murcia
8,38.9,39.3,39.4,39.5,39.7,40.0,40.2,40.3,40.3,archena
9,37.2,37.3,37.6,37.9,38.1,38.4,38.7,39.0,39.3,beniel


In [6]:
# Ruta destino de la Delta Table
delta_path = tfm_utils.full_table_path(table_name)

print(f"Escribiendo {df_normalized.count()} registros en la Delta Table: {table_name}")
print(f"Destino: {delta_path}")

# Escritura en modo overwrite
(df_normalized
    .write
    .mode("overwrite")
    .format("delta")
    .save(delta_path)
)

print("¡Escritura en Delta Lake completada con éxito!")

Escribiendo 45 registros en la Delta Table: raw.crem.edad_media_poblacion
Destino: /tfm/delta_lake/raw/crem/edad_media_poblacion
¡Escritura en Delta Lake completada con éxito!
